# 07 — Computational Cost Evaluation

This includes index build time, retrieval latency, embedding/scoring time, number of documents scored, similarity calculations, peak RSS, iterations to termination, LLM calls, and estimated LLM tokens. Retrieval overhead is computed as (SAR retrieval latency - baseline retrieval latency) / baseline retrieval latency * 100 when a baseline run directory is supplied; it is not treated as the full system cost when methods differ in the number of MIGRES iterations they require.

In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml", ROOT / "configs" / "gemini.yaml"]


In [ ]:
RUN_EXPERIMENT=False
METHOD='sar'
LIMIT=20
RUN_ROOT=ROOT/'outputs/experiments/07_computational_cost'
BASELINE_RUN_DIR=None  # optional completed baseline run for retrieval-overhead comparison


In [ ]:
from sar_migres.utils.config import load_config, validate_canonical_protocol
from sar_migres.datasets.loader import load_phase_questions
from sar_migres.experiments.runner import SARMigresPipeline
from sar_migres.experiments.registry import baseline_overrides
from sar_migres.experiments.parameter_selection import frozen_setting
from sar_migres.experiments.iterative import cost_metrics_from_run
from sar_migres.evaluation.cost import retrieval_overhead_pct
from sar_migres.evaluation.plotting import save_bar_plot
from sar_migres.utils.paths import make_run_id
COST_ROWS={}
base=load_config(*BASE_CONFIG)
validate_canonical_protocol(base)
fp=ROOT/'outputs/experiments/01_parameter_selection/selected_config.json'
DATASETS=list(base.data.datasets)
if RUN_EXPERIMENT:
    K_FIXED, frozen_weights = frozen_setting(fp)
    for DATASET in DATASETS:
        k=K_FIXED
        ov={'retrieval':{'method':METHOD},'sar':{'k':k},'baseline':{'k':k}} if METHOD!='sar' else {'retrieval':{'method':'sar'},'sar':{'k':k,'alpha':frozen_weights.alpha,'beta':frozen_weights.beta,'gamma':frozen_weights.gamma,'weighting':'static'},'baseline':{'k':k}}
        if METHOD!='sar':
            ov=baseline_overrides(METHOD,k=k,run_name=f'cost_{DATASET}_{METHOD}')
        ov['data']={'dataset':DATASET,'dataset_path':base.data.dataset_paths[DATASET],'split':base.data.final_split}
        ov['run']={'name':f'cost_{DATASET}_{METHOD}','output_dir':str(RUN_ROOT),'resume':False}
        items=load_phase_questions(DATASET,base.data.dataset_paths[DATASET],'final',limit=LIMIT,final_split=base.data.final_split,validation_fraction=base.data.validation_fraction,validation_seed=base.data.validation_seed)
        run_id=make_run_id(f'cost_{DATASET}_{METHOD}')
        run_dir=RUN_ROOT/METHOD/'runs'/run_id
        pipe=SARMigresPipeline(load_config(*BASE_CONFIG,overrides=ov),run_id=run_id,run_dir=run_dir).prepare()
        pipe.run_batch(items,resume=False)
        cost=cost_metrics_from_run(pipe.run_dir)
        if BASELINE_RUN_DIR:
            baseline_cost=cost_metrics_from_run(BASELINE_RUN_DIR)
            cost['retrieval_overhead_pct_vs_baseline']=retrieval_overhead_pct(cost['retrieval_latency_total_s'],baseline_cost['retrieval_latency_total_s'])
        COST_ROWS[DATASET]=cost
        out=RUN_ROOT/METHOD/'summaries'/f'{DATASET}_cost.json'
        out.parent.mkdir(parents=True,exist_ok=True)
        out.write_text(json.dumps(cost,indent=2,allow_nan=True),encoding='utf-8')
        print(DATASET,json.dumps(cost,indent=2,allow_nan=True))
else:
    print('Preflight only. Method:',METHOD,'Datasets:',DATASETS,'K:',base.sar.k)
    print('Frozen parameters:', 'present' if fp.exists() else 'MISSING (required before RUN_EXPERIMENT=True)')

if COST_ROWS:
    plot_root=RUN_ROOT/METHOD/'plots'
    save_bar_plot(list(COST_ROWS),[v['retrieval_latency_total_s'] for v in COST_ROWS.values()],'Seconds','Retrieval latency by dataset',plot_root/'retrieval_latency.png')
    save_bar_plot(list(COST_ROWS),[v['llm_calls'] for v in COST_ROWS.values()],'Calls','LLM calls by dataset',plot_root/'llm_calls.png')
    save_bar_plot(list(COST_ROWS),[float((v.get('resource_usage') or {}).get('peak_rss_mb') or 0) for v in COST_ROWS.values()],'MB','Peak process memory by dataset',plot_root/'peak_memory.png')
    save_bar_plot(list(COST_ROWS),[float((v.get('resource_usage') or {}).get('gpu_peak_allocated_mb') or 0) for v in COST_ROWS.values()],'MB','Peak GPU allocation by dataset',plot_root/'gpu_peak_memory.png')
    save_bar_plot(list(COST_ROWS),[v['llm_estimated_tokens'] for v in COST_ROWS.values()],'Estimated tokens','LLM tokens by dataset',plot_root/'llm_tokens.png')
    print('Cost plots:',plot_root)
